In [1]:
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
import os
import struct
from skimage.measure import marching_cubes
from scipy.ndimage import gaussian_filter, label, binary_fill_holes
import scipy.ndimage as ndimage

#Chemins de base
patient = 's0224'
dossier_sortie = '/kaggle/working/'

#Configuration : On définit ici le dossier racine propre à chaque os
config_os = {
    'scapula': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_scapula2.pth',
        'labels_gt': ['scapula_left', 'scapula_right'],
        'chemin_racine': '/kaggle/input/datasets/raphalperon/data-scap/patients4/patients3/patients3'
    },
    'humerus': {
        'modele': '/kaggle/input/datasets/raphalperon/data-scap/meilleur_modele_humerus33.pth',
        'labels_gt': ['humerus_left', 'humerus_right'],
        'chemin_racine': '/kaggle/input/datasets/raphalperon/data-scap/patient/patient' 
    }
}

#Modèle
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class UNet3D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encodeur = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        for feature in features:
            self.encodeur.append(DoubleConv(in_channels, feature))
            in_channels = feature
        self.fond = DoubleConv(features[-1], features[-1] * 2)
        self.decodeur_up = nn.ModuleList()
        self.decodeur_conv = nn.ModuleList()
        for feature in reversed(features):
            self.decodeur_up.append(nn.ConvTranspose3d(feature * 2, feature, kernel_size=2, stride=2))
            self.decodeur_conv.append(DoubleConv(feature * 2, feature))
        self.sortie = nn.Conv3d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skip_connections = []
        for couche in self.encodeur:
            x = couche(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.fond(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.decodeur_up)):
            x = self.decodeur_up[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape: x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decodeur_conv[i](x)
        return self.sigmoid(self.sortie(x))

#Préparation CT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
chemin_ct = os.path.join(config_os['scapula']['chemin_racine'], patient, 'ct.nii')
volume_ct = nib.load(chemin_ct).get_fdata().astype(np.float32)
# Normalisation Hounsfield identique à l'entraînement
volume_ct = np.clip(volume_ct, -200, 1800)
volume_ct = (volume_ct - (-200)) / (1800 - (-200))
z_max, y_max, x_max = volume_ct.shape

def exporter_stl(sommets, faces, chemin_fichier):
    with open(chemin_fichier, 'wb') as fichier:
        fichier.write(b'\0' * 80)
        fichier.write(struct.pack('<I', len(faces)))
        for face in faces:
            s0, s1, s2 = sommets[face[0]], sommets[face[1]], sommets[face[2]]
            normale = np.cross(s1 - s0, s2 - s0)
            norme = np.linalg.norm(normale)
            if norme > 0: normale /= norme
            fichier.write(struct.pack('<3f', *normale))
            fichier.write(struct.pack('<3f', *s0))
            fichier.write(struct.pack('<3f', *s1))
            fichier.write(struct.pack('<3f', *s2))
            fichier.write(struct.pack('<H', 0))

#Boucle d'inférence
for nom_os, config in config_os.items():
    print(f"\n--- Traitement : {nom_os.upper()} ---")

    #Chargement de la vérité terrain (GT) pour évaluation
    dossier_seg = os.path.join(config['chemin_racine'], patient, 'segmentations')
    
    masque_verite = np.zeros(volume_ct.shape, dtype=np.uint8)
    for label_name in config['labels_gt']:
        path_gt = os.path.join(dossier_seg, f'{label_name}.nii')
        if os.path.exists(path_gt):
            m_local = (nib.load(path_gt).get_fdata() > 0).astype(np.uint8)
            if m_local.shape != volume_ct.shape: m_local = np.transpose(m_local, (2, 1, 0))
            masque_verite = np.clip(masque_verite + m_local, 0, 1)
            
    #Chargement du modèle et gestion du mode DataParallel si nécessaire
    modele = UNet3D().to(device)
    poids = torch.load(config['modele'], map_location=device, weights_only=False)
    if any(cle.startswith('module.') for cle in poids.keys()):
        poids = {cle.replace('module.', ''): v for cle, v in poids.items()}
    modele.load_state_dict(poids)
    modele.eval()
    #Inférence par fenêtre glissante (Sliding Window) avec accumulation
    carte_prediction = np.zeros(volume_ct.shape, dtype=np.float32)
    compteur_patches = np.zeros(volume_ct.shape, dtype=np.float32)
    taille, pas = 64, 32

    with torch.no_grad():
        for z in range(0, z_max - taille + 1, pas):
            for y in range(0, y_max - taille + 1, pas):
                for x in range(0, x_max - taille + 1, pas):
                    patch_ct = volume_ct[z:z+taille, y:y+taille, x:x+taille]
                    patch_tensor = torch.tensor(patch_ct).unsqueeze(0).unsqueeze(0).float().to(device)
                    pred_patch = modele(patch_tensor).squeeze().cpu().numpy()
                    carte_prediction[z:z+taille, y:y+taille, x:x+taille] += pred_patch
                    compteur_patches[z:z+taille, y:y+taille, x:x+taille] += 1
                    
    #Normalisation par le nombre de passages par voxel et seuillage
    carte_prediction /= np.maximum(compteur_patches, 1)
    masque_binaire = (carte_prediction > 0.5).astype(np.uint8)
    
    #Post-traitement : extraction de la composante connexe principale et remplissage des cavités
    labels_cc, nb = label(masque_binaire)
    tailles = ndimage.sum(masque_binaire, labels_cc, range(1, nb+1))
    labels_gardes = [idx+1 for idx, t in enumerate(tailles) if t > 10000]
    masque_final = np.isin(labels_cc, labels_gardes).astype(np.uint8)
    masque_final = binary_fill_holes(masque_final).astype(np.uint8)

    #Calcul du dice
    intersection = (masque_final * masque_verite).sum()
    dice = 2 * intersection / (masque_final.sum() + masque_verite.sum())
    print(f"Dice Score {nom_os} : {dice:.3f}")

    #Reconstruction de surface via Marching Cubes et export STL
    masque_lisse = gaussian_filter(masque_final.astype(np.float32), sigma=1.0)
    sommets, faces, _, _ = marching_cubes(masque_lisse, level=0.4)
    chemin_stl = os.path.join(dossier_sortie, f'{nom_os}_{patient}.stl')
    exporter_stl(sommets, faces, chemin_stl)

print("\nTerminé.")


--- Traitement : SCAPULA ---
Dice Score scapula : 0.909

--- Traitement : HUMERUS ---
Dice Score humerus : 0.964

Terminé.
